In [2]:
import pandas as pd

df = pd.read_csv("/Users/milanmanoj/Downloads/tips.csv")
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df_encoded = pd.get_dummies(df, drop_first=True).astype(int)
df_encoded.head()

,total_bill,tip,size,sex_Male,smoker_Yes,day_Sat,day_Sun,day_Thur,time_Lunch
0,16,1,2,0,0,0,1,0,0
1,10,1,3,1,0,0,1,0,0
2,21,3,3,1,0,0,1,0,0
3,23,3,2,1,0,0,1,0,0
4,24,3,4,0,0,0,1,0,0


Behaviour Prediction

In [4]:
x = df_encoded.drop("tip", axis=1)
y = df_encoded["tip"]

In [6]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [9]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

knn = KNeighborsRegressor(n_neighbors=20)

knn.fit(x_train_scaled, y_train)

y_pred = knn.predict(x_test_scaled)

print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R2: 0.41287548855388057
MSE: 0.8759183673469386


In [11]:
df_encoded["predicted_tip"] = knn.predict(scaler.transform(x))
df_encoded["anomaly_score"] = abs(df_encoded["tip"] - df_encoded["predicted_tip"])
df_encoded.sort_values("anomaly_score", ascending=False).head(10)

,total_bill,tip,size,sex_Male,smoker_Yes,day_Sat,day_Sun,day_Thur,time_Lunch,predicted_tip,anomaly_score
170,50,10,3,1,1,1,0,0,0,4.15,5.85
212,48,9,4,1,0,1,0,0,0,4.00,5.00
214,28,6,3,0,1,1,0,0,0,2.50,3.50
23,39,7,4,1,0,1,0,0,0,3.95,3.05
46,22,5,2,1,0,0,1,0,0,2.35,2.65
172,7,5,2,1,1,0,1,0,0,2.45,2.55
88,24,5,2,1,0,0,0,1,1,2.50,2.50
183,23,6,4,1,1,0,1,0,0,3.65,2.35
47,32,6,4,1,0,0,1,0,0,3.65,2.35
73,25,5,2,0,1,1,0,0,0,2.70,2.30


Risk Classification

In [12]:
threshold = df_encoded["tip"].median()
df_encoded["risk_flag"] = (df_encoded["tip"] > threshold).astype(int)

In [14]:
df_encoded.head()

,total_bill,tip,size,sex_Male,smoker_Yes,day_Sat,day_Sun,day_Thur,time_Lunch,predicted_tip,anomaly_score,risk_flag
0,16,1,2,0,0,0,1,0,0,2.25,1.25,0
1,10,1,3,1,0,0,1,0,0,2.05,1.05,0
2,21,3,3,1,0,0,1,0,0,2.65,0.35,1
3,23,3,2,1,0,0,1,0,0,2.35,0.65,1
4,24,3,4,0,0,0,1,0,0,3.65,0.65,1


In [16]:
x = df_encoded.drop(["tip", "risk_flag", "predicted_tip", "anomaly_score"], axis=1)
y = df_encoded["risk_flag"]

In [20]:
x_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42
)
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log_model = LogisticRegression()
log_model.fit(x_train_scaled, y_train)

y_pred = log_model.predict(x_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7142857142857143
              precision    recall  f1-score   support

           0       0.85      0.69      0.76        32
           1       0.57      0.76      0.65        17

    accuracy                           0.71        49
   macro avg       0.71      0.73      0.70        49
weighted avg       0.75      0.71      0.72        49



In [24]:
risk_probability = log_model.predict_proba(x_test_scaled)[:,1]
risk_probability[:10]

array([0.42343606, 0.20473433, 0.86052639, 0.81998161, 0.26788007,
       0.52433628, 0.75502476, 0.21784486, 0.38442149, 0.58640392])

Anomaly Discovery

In [29]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=5, random_state=42)
df_encoded["cluster"] = kmeans.fit_predict(x_cluster_scaled)

In [30]:
df_encoded.groupby("cluster").mean(numeric_only=True)

,total_bill,tip,size,sex_Male,smoker_Yes,day_Sat,day_Sun,day_Thur,time_Lunch,predicted_tip,anomaly_score,risk_flag
cluster,,,,,,,,,,,,
0,22.391304,3.065217,2.913043,0.956522,0.217391,0.934783,0.0,0.000000,0.000000,2.845652,1.065217,0.565217
1,17.735849,2.396226,2.094340,0.415094,0.773585,0.830189,0.0,0.000000,0.000000,2.433962,0.873585,0.452830
2,20.533333,2.920000,2.800000,0.760000,0.253333,0.000000,1.0,0.000000,0.000000,2.875333,0.836667,0.613333
3,32.700000,4.500000,4.900000,0.500000,0.200000,0.000000,0.1,0.900000,0.900000,3.545000,1.045000,0.900000
4,14.616667,2.150000,2.050000,0.483333,0.350000,0.000000,0.0,0.883333,0.983333,2.147500,0.655833,0.266667
